# Liver Segmentation with a Vision Transformer (ViT-UNETR 2.5D)
### A simulated Kaggle competition, run on Google Colab

This notebook imitates the rules of the liver-segmentation competition **without joining it**. The data is the public LiTS dataset (abdominal CT scans + liver masks, NIfTI `.nii`).

| Competition rule | How it is imitated here |
|---|---|
### Training Set
- **80 cases with liver masks:** 80 random LiTS cases (CT + mask).
- **Validation split:** 15% of the training cases are kept aside for validation.

### Test Set
- **Held-out cases, no labels:** All the other LiTS cases (about 50).
- The model only sees their **CT scans**. Their masks are opened only in Section IX to compute the score.

### Data Format
- **NIfTI (`.nii`) volumes:** LiTS data is already provided in `.nii` format.

### Evaluation Metric
- **Dice score:** Calculated over the entire 3D volume and averaged across the test cases.

### Submission Format
- **Prediction masks:** One `(512, 512, n_slice)` mask per CT volume.
- **Encoding:** Run-length encoding (RLE) is applied slice by slice.
- **File:** `submission.csv`, containing columns `id,rle`.
- **Slice IDs:** Formatted as `cell_<idx>_<slice>`.
- The submission file is generated in Section VIII using the competition's `rle_encode` function.

### Public and Private Leaderboard
- **Public phase:** The first half of the test cases.
- **Private phase:** The second half of the test cases.
- Both Dice scores are printed.

The limit of 5 submissions per day is not imitated: you can run the notebook as often as you like.

**Before you run it**
1. **GPU:** Runtime → Change runtime type → GPU.
2. **Kaggle token** (needed to download the data straight from Kaggle): on Kaggle open Settings → API → *Generate New Token*. In Colab click the key icon (Secrets), add a secret named `KAGGLE_API_TOKEN` with that token, and switch **Notebook access** on.

| Section | Goal |
|---|---|
| **I. Setup** | Install, imports, settings (`CFG`). |
| **II. Dataset** | Download LiTS, pair CT + mask, make the 80-train / test split. |
| **III. Preprocessing** | HU windowing, keep the slices around the liver, resize to 256 × 256. |
| **IV. Dataset, 2.5D Input, Augmentation** | 3-slice input, flips, rotations, brightness changes. |
| **V. Model** | ViT-UNETR: pretrained ViT encoder + U-Net decoder. |
| **VI. Loss & Training** | Dice + Cross-Entropy loss, AdamW, cosine LR, mixed precision. |
| **VII. Predicting Whole Volumes** | Predict full CT volumes at the original size, check Dice on the validation cases. |
| **VIII. Test Prediction & Submission** | Predict the test CTs, RLE-encode every slice, write `submission.csv`. |
| **IX. Scoring** | Decode `submission.csv` and score it against the hidden test masks (Public / Private Dice). |

In [3]:
!pip install -q nibabel timm kagglehub

## I. Setup & Configuration

In [4]:
import os
import re
import glob
import time
import random
import warnings

import numpy as np
import pandas as pd
import cv2
import nibabel as nib
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage as ndi
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

class CFG:
    seed       = 42
    n_train    = 80                    # competition rule: 80 training cases with masks (all other cases = hidden test set)
    val_frac   = 0.15                  # share of the 80 training cases used for validation

    img_size   = 256                   # every slice is resized to 256 x 256
    hu_min, hu_max = -100, 250         # CT window (soft tissue), in Hounsfield Units
    margin     = 15                     # extra slices kept around the liver (training only)
    in_ch      = 3                     # 2.5D input: slices z-1, z, z+1

    vit_name   = "vit_small_patch16_224.augreg_in21k_ft_in1k"
    batch_size = 16
    epochs     = 30
    lr_vit_enc = 1e-4                  # pretrained ViT encoder
    lr_vit_dec = 5e-4                  # new U-Net decoder
    weight_decay = 1e-2
    grad_clip  = 1.0                   # NEW: limit the size of one update step
    num_workers = 2                    # NEW: load / augment the next batch on the CPU while the GPU is busy


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print(f"torch: {torch.__version__} | timm: {timm.__version__} | device: {DEVICE}")
if DEVICE.type == "cpu":
    print("WARNING: no GPU found. Runtime -> Change runtime type -> GPU")

torch: 2.11.0+cu130 | timm: 1.0.30 | device: cuda


## II. Dataset

Kaggle hosts LiTS as two datasets, and both are needed to pair every CT with its mask:
- [Part 1](https://www.kaggle.com/datasets/andrewmvd/liver-tumor-segmentation): the masks (`segmentations/`) + the CTs `volume_pt1` … `volume_pt5`.
- [Part 2](https://www.kaggle.com/datasets/andrewmvd/liver-tumor-segmentation-part-2): the remaining CTs (`volume_pt6` onwards).

The dataset is formatted in the standard 3D medical imaging format **NIfTI (`.nii`)**, structured into two primary file groups:

- CT Scan Volumes (`volume-*.nii`)
    - Description: 3D Computed Tomography scans of patients.
    - Dimensions: Spatial shape $H \times W \times Z$ (typically $512 \times 512 \times Z$, where $Z$ denotes the number of axial slices).
    - Pixel Values: Represent tissue density in **Hounsfield Units (HU)**, ranging from $-1000$ (air) to $+3000$ (bone).


- Segmentation Masks (`segmentation-*.nii`)
    - Description: 3D ground-truth annotations matching the exact spatial dimensions of their corresponding volume files. **Class 0 — Background:** Non-liver structures, including air, lungs, bones, and adjacent organs. **Class 1 — Liver:** Healthy liver tissue. **Class 2 — Tumor:** Liver tumor regions.

In [5]:
import kagglehub

part1 = kagglehub.dataset_download("andrewmvd/liver-tumor-segmentation")           # masks + volume_pt1 ... pt5
part2 = kagglehub.dataset_download("andrewmvd/liver-tumor-segmentation-part-2")    # volume_pt6 onwards
print(part1)
print(part2)

100%|██████████| 4.84G/4.84G [04:44<00:00, 18.2MB/s]

Extracting files...


100%|██████████| 10.8G/10.8G [10:46<00:00, 18.0MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/andrewmvd/liver-tumor-segmentation/versions/5
/root/.cache/kagglehub/datasets/andrewmvd/liver-tumor-segmentation-part-2/versions/2


**Competition Split.** A CT and its mask are paired through the number in the file name (`volume-12.nii` ↔ `segmentation-12.nii`). The matched cases are shuffled and divided into:
- **80 training cases** (CT + mask). The model trains on them; 15% of them are used for validation.
- **Test cases** (remaning). Only their CTs are used to predict; their masks stay unopened until Section IX.


In [7]:
def case_id(path):
    """Case number = last number in the file name, e.g. volume-12.nii -> 12"""
    return int(re.findall(r"\d+", os.path.basename(path).split(".")[0])[-1])

files = sorted(glob.glob(f"{part1}/**/*.nii*", recursive=True) + glob.glob(f"{part2}/**/*.nii*", recursive=True))
CT_FILES   = {case_id(p): p for p in files if os.path.basename(p).startswith("volume")}         # volume-<id>.nii
MASK_FILES = {case_id(p): p for p in files if os.path.basename(p).startswith("segmentation")}   # segmentation-<id>.nii
CASE_IDS = sorted(set(CT_FILES) & set(MASK_FILES))        # a case is used only if both its CT and its mask exist

print(f"Total NIfTI files: {len(files)} | CT: {len(CT_FILES)} | masks: {len(MASK_FILES)}")
assert len(CASE_IDS) > CFG.n_train, "Not enough CT/mask pairs - check that BOTH datasets were downloaded."

ids = np.array(CASE_IDS)
np.random.RandomState(CFG.seed).shuffle(ids)
TRAIN_POOL = ids[:CFG.n_train].tolist()            # the 80 "training set" cases (CT + mask)
TEST_IDS   = sorted(ids[CFG.n_train:].tolist())    # the held-out "test set": only their CTs are used for prediction

n_val = int(CFG.val_frac * len(TRAIN_POOL))        # validation patients are taken from the 80 training cases
VAL_IDS, TRAIN_IDS = TRAIN_POOL[:n_val], TRAIN_POOL[n_val:]
print(f"Number of Training set: {len(TRAIN_POOL)} cases (train: {len(TRAIN_IDS)} | validation: {len(VAL_IDS)})")
print(f"Number of hidden Test set: {len(TEST_IDS)} cases")

Total NIfTI files: 262 | CT: 131 | masks: 131
Number of Training set: 80 cases (train 68 | val 12)
Number of hidden Test set: 51 cases


In [8]:
# One training code instance
ct = nib.load(CT_FILES[TRAIN_POOL[0]])
mask = np.asanyarray(nib.load(MASK_FILES[TRAIN_POOL[0]]).dataobj)
print("------Information about one training instance------")
print("Shape:", ct.shape, "| Voxel Size mm:", ct.header.get_zooms()[:3], "| Orientation:", nib.aff2axcodes(ct.affine))
print("Mask values:", np.unique(mask), "(LiTS: 0 background, 1 liver, 2 tumour)")

------Information about one training instance------
Shape: (512, 512, 192) | Voxel Size mm: (np.float32(0.666016), np.float32(0.666016), np.float32(2.5)) | Orientation: ('L', 'P', 'S')
Mask values: [0 1 2] (LiTS: 0 background, 1 liver, 2 tumour)


## IV. Static Dataset Preprocessing
### 1. Hounsfield Unit (HU) Windowing & Normalization
* **Rationale:** Raw CT HU values contain irrelevant background structures (bones, air).
* **Execution:** Intensity values are clipped to the liver window range **$[-100, 250]$ HU** (or $[-100, 400]$ HU) to highlight soft tissue contrast.
* **Normalization:** Clipped HU values are linearly rescaled to the $[0, 1]$ interval:
  $$\text{Pixel}_{\text{norm}} = \frac{\text{Pixel} - \text{HU}_{\min}}{\text{HU}_{\max} - \text{HU}_{\min}}$$

### 2. Slicing & Noise Filtering
* **3D to 2D / 2.5D Conversion:** 3D volumes are sliced axially into 2D images or stacked with neighboring slices (2.5D context) to preserve volumetric depth information.
* **Filtering Empty Slices:** Slices containing only background pixels are filtered out to optimize training efficiency, keeping only slices containing Liver (`1`) or Tumor (`2`) labels.

### 3. Spatial Resizing
* Slice dimensions ($512 \times 512$) are resized to fixed square resolutions such as **$224 \times 224$** or **$256 \times 256$** to match the patch size requirements of Vision Transformer backbones (e.g., Swin UNETR, ViT).

### 4. Data Augmentation
* Spatial transformations (Random Rotation, Random Flipping) and Gaussian noise injection are applied to mitigate overfitting on limited medical training datasets.

### 5. Tensor Formatting & Data Loaders
* **Tensor Formatting:** Processed NumPy arrays are converted to PyTorch Tensors with shape $(B, C, H, W)$.
* **Lazy Loading:** Data is loaded dynamically via PyTorch/MONAI Datasets during batch generation to prevent System RAM Out-Of-Memory (OOM) errors.

In [9]:
def window_hu(hu_slice, hu_min=CFG.hu_min, hu_max=CFG.hu_max):
    hu_clipped = np.clip(hu_slice, hu_min, hu_max)
    hu_norm = (hu_clipped - hu_min) / (hu_max - hu_min) * 255.0
    return hu_norm.astype(np.uint8)

def reorient_to_zyx(arr):
    return np.ascontiguousarray(np.moveaxis(np.rot90(arr), 2, 0))

def preprocess_case(cid):
    vol = reorient_to_zyx(np.asanyarray(nib.load(CT_FILES[cid]).dataobj))
    lab = reorient_to_zyx(np.asanyarray(nib.load(MASK_FILES[cid]).dataobj) > 0).astype(np.uint8)   # liver = 1

    if vol.shape != lab.shape:
        return None

    zs = np.where(lab.any(axis=(1, 2)))[0]            # slices that contain liver
    if len(zs) == 0:
        return None

    z0 = max(zs.min() - CFG.margin, 0)
    z1 = min(zs.max() + CFG.margin + 1, len(lab))

    S = CFG.img_size
    img = np.array([cv2.resize(window_hu(vol[z]), (S, S), interpolation=cv2.INTER_AREA) for z in range(z0, z1)], dtype=np.uint8)
    msk = np.array([cv2.resize(lab[z], (S, S), interpolation=cv2.INTER_NEAREST) for z in range(z0, z1)], dtype=np.uint8)
    return {"img": img, "lab": msk}

DATA = {}
for cid in tqdm(TRAIN_POOL, desc="Processing CT scans"):
    case_data = preprocess_case(cid)
    if case_data is not None:
        DATA[cid] = case_data

print(f"Successfully processed {len(DATA)} cases. Total slices kept: {sum(len(d['img']) for d in DATA.values()):,}")

Processing CT scans:   0%|          | 0/80 [00:00<?, ?it/s]

Successfully processed 80 cases. Total slices kept: 13,982


## IV. Dynamic Data Augmentation

1. **2.5D Volumetric Context Construction**
   - Stacking three consecutive slices $(z-1, z, z+1)$ along the channel dimension, thus generate a 3-channel input tensor with shape $(3, H, W)$, helping the model distinguish continuous tumor boundaries from isolated cross-sectional blood vessels.
   - Matches the 3-channel (RGB) input shape required by ImageNet-pretrained Vision Transformer (ViT) encoders.

2. **Targeted Data Augmentation (`augment`)**
   - **Random Horizontal Flip:** Applied with $50\%$ probability to both the CT slice and the segmentation mask.
   - **Small Spatial Rotation:** Applied with a $40\%$ probability by rotating CT slice stacks using an affine transform matrix (`cv2.getRotationMatrix2D`).
   - **Intensity & Contrast Adjustment:** Applied with a $40\%$ probability by scaling pixel contrast ($\alpha \in [0.9, 1.1]$) and shifting brightness ($\beta \in [-10, 10]$) only on CT channels.

In [10]:
def augment(x, y):
    C, H, W = x.shape

    if random.random() < 0.5:                         # horizontal flip
        x = np.ascontiguousarray(np.flip(x, axis=2))
        y = np.ascontiguousarray(np.flip(y, axis=1))

    if random.random() < 0.6:                         # CHANGED: rotation + zoom + shift in ONE affine transform (was rotation only, 40%)
        angle = random.uniform(-15, 15)
        scale = random.uniform(0.85, 1.15)            # NEW: zoom (patients have different body sizes / fields of view)
        M = cv2.getRotationMatrix2D((W / 2, H / 2), angle, scale)
        M[0, 2] += random.uniform(-0.08, 0.08) * W    # NEW: shift left / right by up to 8 %
        M[1, 2] += random.uniform(-0.08, 0.08) * H    # NEW: shift up / down by up to 8 %
        x = np.array([
            cv2.warpAffine(np.ascontiguousarray(x[c]), M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
            for c in range(C)
        ])
        y = cv2.warpAffine(np.ascontiguousarray(y), M, (W, H), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    if random.random() < 0.5:                         # brightness / contrast (CHANGED: 40% -> 50%)
        alpha = random.uniform(0.9, 1.1)
        beta = random.uniform(-10, 10)
        x = np.clip(x.astype(np.float32) * alpha + beta, 0, 255)

    if random.random() < 0.3:                         # NEW: Gaussian noise (the text above promised it, but the code did not have it)
        sigma = random.uniform(2, 8)                  # on the 0-255 grey scale
        x = np.clip(x.astype(np.float32) + np.random.normal(0, sigma, x.shape), 0, 255)

    return x, np.ascontiguousarray(y)

In [11]:
class LiverDataset(Dataset):
    def __init__(self, case_ids, is_train=False):
        self.cases = [DATA[c] for c in case_ids if c in DATA]
        self.is_train = is_train
        self.samples = [(i, z) for i, case in enumerate(self.cases) for z in range(len(case["img"]))]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        i, z = self.samples[idx]
        img, lab = self.cases[i]["img"], self.cases[i]["lab"]

        x = np.stack([img[max(0, z - 1)], img[z], img[min(len(img) - 1, z + 1)]], axis=0)     # 2.5D stack
        y = lab[z]

        if self.is_train:
            x, y = augment(x, y)

        x = (x.astype(np.float32) / 255.0 - 0.5) / 0.5
        return torch.from_numpy(np.ascontiguousarray(x)).float(), torch.from_numpy(np.ascontiguousarray(y)).long()


train_ds = LiverDataset(TRAIN_IDS, is_train=True)
val_ds = LiverDataset(VAL_IDS)

loader_args = dict(num_workers=CFG.num_workers, persistent_workers=CFG.num_workers > 0, pin_memory=USE_AMP)
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, drop_last=True, pin_memory=USE_AMP, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, pin_memory=USE_AMP, num_workers=0)

print(f"Total Train slices: {len(train_ds):,} | Validation slices: {len(val_ds):,}")

Total Train slices: 11,808 | Validation slices: 2,174


## V. Model Architecture – **ViT-UNETR (2.5D)**

A pretrained **Vision Transformer (ViT-Small)** encoder plus a **U-Net decoder**. Input: 3-slice stack $(B, 3, 256, 256)$. Output: 2 logits per pixel $(B, 2, 256, 256)$ (background, liver).

**Encoder (ViT):** loaded with `timm` (ImageNet-pretrained, `img_size=256`, classification head removed). The image is cut into $16 \times 16$ patches, so the transformer works on a $16 \times 16$ grid with 384 channels. The model reads the output of the transformer blocks 3, 6, 9 and 12 (`self.taps`) as four feature maps.

**Decoder (U-Net):**
- `ConvBlock` = two `3×3 Conv -> BatchNorm -> ReLU`.
- The deepest ViT feature goes through a bottleneck, then four `ConvTranspose2d` steps double the size each time ($16 \to 32 \to 64 \to 128 \to 256$).
- At each step the result is concatenated with a skip connection (`up_chain` upsamples the ViT features to the matching size; `stem` is a `ConvBlock` on the raw input for the final full-resolution step).
- `head` (`1×1 Conv`) outputs the 2 class logits.

In [12]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.net(x)


def up_chain(in_channels, out_channels, num_upsamples):
    layers = []
    curr_ch = in_channels
    for _ in range(num_upsamples):
        layers.extend([
            nn.ConvTranspose2d(curr_ch, out_channels, kernel_size=2, stride=2),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        ])
        curr_ch = out_channels
    return nn.Sequential(*layers)


def create_vit(name, img_size, in_ch):
    return timm.create_model(name, pretrained=True, img_size=img_size, in_chans=in_ch, num_classes=0)


class ViTUNETR(nn.Module):
    def __init__(self, vit_name=CFG.vit_name, img_size=CFG.img_size, in_ch=CFG.in_ch, n_classes=2):
        super().__init__()
        self.vit = create_vit(vit_name, img_size, in_ch)
        embed_dim = self.vit.embed_dim
        depth = len(self.vit.blocks)
        self.taps = [depth // 4 - 1, depth // 2 - 1, 3 * depth // 4 - 1, depth - 1]

        # NEW: small CNN on the raw slices -> sharp edges. stem = full resolution, cnn1 = 1/2, cnn2 = 1/4
        self.stem = ConvBlock(in_ch, 16)
        self.cnn1 = ConvBlock(16, 32)
        self.cnn2 = ConvBlock(32, 64)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(embed_dim, 256)

        self.up3   = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.skip2 = up_chain(embed_dim, 128, num_upsamples=1)
        self.dec3  = ConvBlock(128 + 128, 128)

        self.up2   = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.skip1 = up_chain(embed_dim, 64, num_upsamples=2)
        self.dec2  = ConvBlock(64 + 64 + 64, 64)          # CHANGED: + 64 channels from cnn2

        self.up1   = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.skip0 = up_chain(embed_dim, 32, num_upsamples=3)
        self.dec1  = ConvBlock(32 + 32 + 32, 32)          # CHANGED: + 32 channels from cnn1

        self.up0   = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec0  = ConvBlock(16 + 16, 16)

        self.head  = nn.Conv2d(16, n_classes, kernel_size=1)

    def forward(self, x):
        f0, f1, f2, f3 = self.vit.get_intermediate_layers(x, n=self.taps, reshape=True)
        s0 = self.stem(x)                                 # (B, 16, S,   S)
        s1 = self.cnn1(self.pool(s0))                     # (B, 32, S/2, S/2)
        s2 = self.cnn2(self.pool(s1))                     # (B, 64, S/4, S/4)
        y = self.bottleneck(f3)                                                       # S/16
        y = self.dec3(torch.cat([self.up3(y), self.skip2(f2)], dim=1))                # S/8
        y = self.dec2(torch.cat([self.up2(y), self.skip1(f1), s2], dim=1))            # S/4
        y = self.dec1(torch.cat([self.up1(y), self.skip0(f0), s1], dim=1))            # S/2
        y = self.dec0(torch.cat([self.up0(y), s0], dim=1))                            # S
        return self.head(y)


def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

model = ViTUNETR().to(DEVICE)
with torch.no_grad():
    dummy = torch.randn(2, CFG.in_ch, CFG.img_size, CFG.img_size, device=DEVICE)
    out = model(dummy)

print(f"Successfully initialize the ViTUNETR model on {DEVICE}!")
print(f"Output size: {tuple(out.shape)} | Total parameters: {count_params(model):.2f}M")

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Successfully initialize the ViTUNETR model on cuda!
Output size: (2, 2, 256, 256) | Total parameters: 24.42M


## VI. Loss & Training

1. **Loss (`DiceCELoss`):** `0.5 * Cross-Entropy + 0.5 * soft Dice`. Cross-Entropy looks at every pixel; the soft Dice term measures the overlap with the liver mask directly (the same idea as the competition metric).
2. **Validation metric:** hard Dice for the liver over all validation slices (`argmax` of the logits).
3. **Optimizer:** `AdamW` with a smaller learning rate (`1e-4`) for the pretrained ViT and a larger one (`5e-4`) for the new decoder.
4. **Schedule:** cosine decay of the learning rate to `1e-6`; mixed precision (AMP) on the GPU for speed.

In [13]:
class DiceCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets)

        liver_prob = F.softmax(logits, dim=1)[:, 1]          # (B, H, W)
        liver_true = (targets == 1).float()
        inter = (liver_prob * liver_true).sum(dim=(1, 2))
        union = liver_prob.sum(dim=(1, 2)) + liver_true.sum(dim=(1, 2))
        dice_loss = 1.0 - ((2.0 * inter + self.smooth) / (union + self.smooth)).mean()

        return 0.5 * ce_loss + 0.5 * dice_loss


@torch.no_grad()
def evaluate_loader(model, loader, criterion):
    model.eval()
    total_loss, inter, union = 0.0, 0.0, 0.0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y)

        total_loss += loss.item()
        pred, true = (logits.argmax(dim=1) == 1), (y == 1)
        inter += (pred & true).sum().item()
        union += pred.sum().item() + true.sum().item()

    dice_liver = (2.0 * inter + 1e-6) / (union + 1e-6)
    return total_loss / max(len(loader), 1), dice_liver


def train_vit_unetr(model, train_loader, val_loader, epochs=CFG.epochs):
    criterion = DiceCELoss().to(DEVICE)
    enc_params = list(model.vit.parameters())
    enc_ids = {id(p) for p in enc_params}
    dec_params = [p for p in model.parameters() if id(p) not in enc_ids]

    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": CFG.lr_vit_enc},
        {"params": dec_params, "lr": CFG.lr_vit_dec}
    ], weight_decay=CFG.weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

    for ep in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        tr_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)                                            # NEW: gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()                                                      # CHANGED: once per batch (was once per epoch)
            tr_loss += loss.item()

        scheduler.step()
        tr_loss /= len(train_loader)
        val_loss, d_liver = evaluate_loader(model, val_loader, criterion)

        dt = time.time() - t0
        print(f"Epoch {ep:02d}/{epochs:02d} [{dt:4.1f}s] | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | Liver Dice: {d_liver:.4f}")


train_vit_unetr(model, train_loader, val_loader)

Epoch 01/30 [83.6s] | Train Loss: 0.4234 | Val Loss: 0.2405 | Liver Dice: 0.9486
Epoch 02/30 [82.5s] | Train Loss: 0.1699 | Val Loss: 0.1411 | Liver Dice: 0.9533
Epoch 03/30 [82.1s] | Train Loss: 0.1212 | Val Loss: 0.1213 | Liver Dice: 0.9507
Epoch 04/30 [82.2s] | Train Loss: 0.0658 | Val Loss: 0.0828 | Liver Dice: 0.9534
Epoch 05/30 [81.4s] | Train Loss: 0.0439 | Val Loss: 0.0533 | Liver Dice: 0.9584
Epoch 06/30 [81.6s] | Train Loss: 0.0401 | Val Loss: 0.0591 | Liver Dice: 0.9545
Epoch 07/30 [81.6s] | Train Loss: 0.0382 | Val Loss: 0.0567 | Liver Dice: 0.9566
Epoch 08/30 [81.2s] | Train Loss: 0.0360 | Val Loss: 0.0567 | Liver Dice: 0.9535
Epoch 09/30 [81.5s] | Train Loss: 0.0354 | Val Loss: 0.0490 | Liver Dice: 0.9599
Epoch 10/30 [81.8s] | Train Loss: 0.0347 | Val Loss: 0.0474 | Liver Dice: 0.9581
Epoch 11/30 [82.4s] | Train Loss: 0.0322 | Val Loss: 0.0667 | Liver Dice: 0.9525
Epoch 12/30 [82.6s] | Train Loss: 0.0321 | Val Loss: 0.0585 | Liver Dice: 0.9586
Epoch 13/30 [82.4s] | Train 

## VII. Predicting Whole Volumes

The validation Dice above is computed on small, cropped 256 × 256 slices. The competition works differently: **whole volumes, original size, every slice**. `predict_case(path)` does exactly that, and the same function is used here (on the validation cases, where we know the answer) and in Section VIII (on the test cases):

1. Read the CT file, apply the same re-orientation, HU window and resize as in training.
2. Build the 3-slice stacks and predict the liver probability, averaged with a horizontally flipped copy (**flip TTA**).
3. **Clean-up:** keep only the largest connected 3D blob of liver. The model was trained only on slices around the liver, so on other slices (chest, pelvis) it can create small false positives; this removes them.
4. Resize the probability back to the original slice size, threshold at 0.5, and undo the re-orientation, so the result has the **same shape and orientation as the input file**, i.e. `(512, 512, n_slice)`.

`dice()` compares a prediction with the real mask. A Dice around 0.9+ on the validation cases also confirms that the re-orientation is undone correctly.

In [14]:
@torch.no_grad()
def predict_case(path):
    """CT file -> 0/1 liver mask with the same shape and orientation as the file (what the competition expects)."""
    model.eval()
    vol = reorient_to_zyx(np.asanyarray(nib.load(path).dataobj))        # (Z, h, w)
    Z, h, w = vol.shape
    S = CFG.img_size
    img = np.array([cv2.resize(window_hu(s), (S, S), interpolation=cv2.INTER_AREA) for s in vol])    # (Z, S, S)

    # 2.5D stacks (z-1, z, z+1), scaled to [-1, 1] like in training
    stacks = [np.stack([img[max(0, z - 1)], img[z], img[min(Z - 1, z + 1)]]) for z in range(Z)]
    x_all = torch.from_numpy((np.array(stacks, dtype=np.float32) / 255.0 - 0.5) / 0.5).float()

    probs = []
    for i in range(0, Z, 32):
        b = x_all[i:i + 32].to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            p = 0.5 * (F.softmax(model(b), 1) + torch.flip(F.softmax(model(torch.flip(b, [3])), 1), [3]))
        probs.append(p[:, 1].float().cpu().numpy())
    prob = np.concatenate(probs)                                         # (Z, S, S) liver probability

    # keep only the largest connected liver blob
    mask = prob > 0.5
    blobs, n = ndi.label(mask)
    if n > 1:
        sizes = ndi.sum(mask, blobs, range(1, n + 1))
        prob[mask & (blobs != np.argmax(sizes) + 1)] = 0

    # back to the original slice size, then undo reorient_to_zyx
    pred = np.array([cv2.resize(p, (w, h), interpolation=cv2.INTER_LINEAR) > 0.5 for p in prob], dtype=np.uint8)   # (Z, h, w)
    return np.rot90(np.moveaxis(pred, 0, 2), k=-1)                       # (H, W, Z), same as the file


def dice(pred, gt):
    p, g = pred > 0, gt > 0
    denom = p.sum() + g.sum()
    return 1.0 if denom == 0 else float(2.0 * (p & g).sum() / denom)


scores = {}
for cid in tqdm(VAL_IDS, desc="Predicting validation volumes"):
    pred = predict_case(CT_FILES[cid])
    gt = np.asanyarray(nib.load(MASK_FILES[cid]).dataobj) > 0
    assert pred.shape == gt.shape, f"shape mismatch: prediction {pred.shape} vs mask {gt.shape}"
    scores[cid] = dice(pred, gt)

val_dice = pd.Series(scores, name="Liver Dice")
print(val_dice.round(4).to_string())
print(f"\nValidation set ({len(val_dice)} patients) - Liver Dice: {val_dice.mean():.4f} ± {val_dice.std():.4f}")

Predicting validation volumes:   0%|          | 0/12 [00:00<?, ?it/s]

55     0.9069
40     0.9718
19     0.9719
31     0.9664
115    0.9472
56     0.8913
69     0.9672
105    0.9742
81     0.9585
26     0.9707
95     0.9308
27     0.9672

Validation set (12 patients) - Liver Dice: 0.9520 ± 0.0279


## VIII. Test Prediction & Submission

**Submission format:** one row per slice of every test volume, `id,rle`.
- `id` = `cell_<idx>_<slice>`: `idx` is the number of the test file (1, 2, 3, …) and slices are counted from 0, as in the example `cell_1_0`, `cell_1_1`, …
- `rle` = run-length encoding of the 512 × 512 slice with the `rle_encode` function from the competition page (column-major order; an empty mask is written as `1 0`).
- The prediction of each volume must have the shape `(512, 512, n_slice)`, exactly like the input CT.

Only the **CT** of each test case is read here; the test masks are not touched.

In [15]:
def rle_encode(mask: np.ndarray) -> str:
    """
    Run-length encode a 2D binary mask (1 for foreground, 0 for background)
    Empty mask -> '1 0'.
    """
    assert mask.ndim == 2, "rle_encode expects a 2D mask"
    pixels = mask.astype(np.uint8).flatten(order='F')  # column-major
    if pixels.max() == 0:
        return "1 0"
    # Pad with zeros at both ends to catch transitions cleanly
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(map(str, runs))


rows = []
for k, cid in enumerate(tqdm(TEST_IDS, desc="Predicting test volumes"), start=1):     # k = number of the test file
    pred = predict_case(CT_FILES[cid])                                                # (512, 512, n_slice)
    assert pred.shape == nib.load(CT_FILES[cid]).shape and pred.shape[:2] == (512, 512), f"bad shape {pred.shape}"
    for z in range(pred.shape[2]):
        rows.append((f"cell_{k}_{z}", rle_encode(pred[:, :, z])))

submission = pd.DataFrame(rows, columns=["id", "rle"])
submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv: {len(submission):,} rows from {len(TEST_IDS)} test volumes")
print(submission.head())

Predicting test volumes:   0%|          | 0/51 [00:00<?, ?it/s]

Saved submission.csv: 23,810 rows from 51 test volumes
         id  rle
0  cell_1_0  1 0
1  cell_1_1  1 0
2  cell_1_2  1 0
3  cell_1_3  1 0
4  cell_1_4  1 0


## IX. Scoring the Submission (Public / Private)

On Kaggle this step is done by the platform. Here we do it ourselves: `rle_decode` turns every `rle` string of `submission.csv` back into a 512 × 512 mask, the slices are stacked into volumes, and each volume is compared with its **hidden mask** (opened here for the first time) using the Dice score.

The test cases are split in half, as in the Public / Private phases: the first half gives the **Public** score, the second half the **Private** score. Each score is the mean of the per-volume Dice.

In [16]:
def rle_decode(rle, shape=(512, 512)):
    """Inverse of rle_encode: 'start length start length ...' (1-based, column-major) -> 2D 0/1 mask"""
    nums = list(map(int, rle.split()))
    flat = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for start, length in zip(nums[0::2], nums[1::2]):
        flat[start - 1:start - 1 + length] = 1
    return flat.reshape(shape, order="F")


sub = pd.read_csv("submission.csv")
rle_of = dict(zip(sub["id"], sub["rle"]))

scores = {}
for k, cid in enumerate(tqdm(TEST_IDS, desc="Scoring"), start=1):
    gt = np.asanyarray(nib.load(MASK_FILES[cid]).dataobj) > 0                          # hidden test mask: used ONLY here
    pred = np.stack([rle_decode(rle_of[f"cell_{k}_{z}"]) for z in range(gt.shape[2])], axis=2)
    scores[f"cell_{k}"] = dice(pred, gt)

scores = pd.Series(scores, name="Liver Dice")
half = len(scores) // 2
print(scores.round(4).to_string())
print(f"\nPublic  score ({half} test cases)                 : {scores.iloc[:half].mean():.4f}")
print(f"Private score ({len(scores) - half} test cases)                 : {scores.iloc[half:].mean():.4f}")

Scoring:   0%|          | 0/51 [00:00<?, ?it/s]

cell_1     0.9705
cell_2     0.9576
cell_3     0.9784
cell_4     0.9665
cell_5     0.9652
cell_6     0.9525
cell_7     0.9648
cell_8     0.9752
cell_9     0.9740
cell_10    0.9606
cell_11    0.9620
cell_12    0.9185
cell_13    0.9335
cell_14    0.9453
cell_15    0.9641
cell_16    0.9564
cell_17    0.9614
cell_18    0.9750
cell_19    0.9522
cell_20    0.9589
cell_21    0.9681
cell_22    0.9418
cell_23    0.9286
cell_24    0.9723
cell_25    0.9537
cell_26    0.9502
cell_27    0.9686
cell_28    0.9700
cell_29    0.9532
cell_30    0.9650
cell_31    0.9832
cell_32    0.9729
cell_33    0.9802
cell_34    0.9723
cell_35    0.9714
cell_36    0.9745
cell_37    0.9651
cell_38    0.9501
cell_39    0.9664
cell_40    0.9764
cell_41    0.9803
cell_42    0.9787
cell_43    0.9802
cell_44    0.9673
cell_45    0.9758
cell_46    0.9176
cell_47    0.9806
cell_48    0.9750
cell_49    0.9770
cell_50    0.9718
cell_51    0.9501

Public  score (25 test cases)                 : 0.9583
Private score (26 test cas